# nnU-Net BraTS 2024 MEN-RT — Google Colab Pipeline

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells **in order**, top to bottom
3. Keep the browser tab open during training

**All configurable parameters are in Cell 3 only.**

| Cell | Step | Time |
|---|---|---|
| 1 | GPU check | 10 sec |
| 2 | Google Drive mount | 30 sec |
| 3 | Environment setup | 5 sec |
| 4 | Clone repo + install | 3 min |
| 5 | Register custom trainer | 10 sec |
| 6 | Set working directory | 5 sec |
| 7 | Prepare dataset | 2-4 min |
| 8 | Verify dataset | 10 sec |
| 9 | Preprocessing | 5 min |
| 10 | **Train all 5 folds** | ~3-4 hours |
| 11 | Training history | 10 sec |
| 12 | Verify checkpoints | 10 sec |
| 13 | Inference | 30 min |
| 14 | Post-processing | 10 min |
| 15 | Evaluation | 10 min |
| 16 | Visualizations | 5 min |
| 17 | Show results | 10 sec |

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, torch

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(smi.stdout)

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
Path('/content/drive/MyDrive/nnunet_checkpoints').mkdir(parents=True, exist_ok=True)
print('Drive mounted. Checkpoints → /content/drive/MyDrive/nnunet_checkpoints')

In [ ]:
# ── Cell 3: Configure Environment ────────────────────────────────────────────
# ALL configurable parameters are here. Edit ONLY this cell.

import os, sys
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT      = '/content/nnunet'
DATA_DIR     = '/content/data/BraTS-MEN-RT-Train-v2'
DRIVE_OUTPUT = '/content/drive/MyDrive/nnunet_checkpoints'

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_ID       = '001'
DATASET_NAME     = 'BraTSMENRT'
LABEL_SUFFIX     = 'gtv'
MAX_TRAIN_CASES  = 20

# ── Training ──────────────────────────────────────────────────────────────────
NNUNET_CONFIGURATION = '3d_fullres'
NNUNET_TRAINER_CLASS = 'nnUNetTrainerEarlyStopping'
NNUNET_PLANS         = 'nnUNetPlans'
NUM_FOLDS            = 5
NNUNET_SEED          = 42
NNUNET_NUM_EPOCHS    = 10
RESUME_TARGET_EPOCHS = 50

# ── Early stopping ────────────────────────────────────────────────────────────
ES_PATIENCE  = 50
ES_MIN_DELTA = 0.0001
ES_WARMUP    = 50

# ── Hardware ──────────────────────────────────────────────────────────────────
CUDA_VISIBLE_DEVICES = '0'
NNUNET_N_PROC_DA     = '2'
NNUNET_COMPILE       = 'false'

# ── Inference ─────────────────────────────────────────────────────────────────
INFERENCE_STEP_SIZE   = 0.5
INFERENCE_DISABLE_TTA = 'false'
INFERENCE_DEVICE      = 'cuda'

# ── Evaluation ────────────────────────────────────────────────────────────────
PP_THRESHOLDS      = '10,25,50,100,200,500'
NSD_TOLERANCE_MM   = 2.0
LOW_DICE_THRESHOLD = 0.5
BOOTSTRAP_N        = 2000

# ─────────────────────────────────────────────────────────────────────────────
os.environ.update({
    'nnUNet_raw'             : f'{PROJECT}/nnunet_raw',
    'nnUNet_preprocessed'    : f'{PROJECT}/nnunet_preprocessed',
    'nnUNet_results'         : f'{PROJECT}/nnunet_results',
    'DATASET_ID'             : DATASET_ID,
    'DATASET_NAME'           : DATASET_NAME,
    'LABEL_SUFFIX'           : LABEL_SUFFIX,
    'MAX_TRAIN_CASES'        : str(MAX_TRAIN_CASES),
    'TRAIN_SOURCE'           : DATA_DIR,
    'VAL_SOURCE'             : '',
    'NNUNET_CONFIGURATION'   : NNUNET_CONFIGURATION,
    'NNUNET_TRAINER_CLASS'   : NNUNET_TRAINER_CLASS,
    'NNUNET_PLANS_IDENTIFIER': NNUNET_PLANS,
    'NUM_FOLDS'              : str(NUM_FOLDS),
    'NNUNET_SEED'            : str(NNUNET_SEED),
    'NNUNET_NUM_EPOCHS'      : str(NNUNET_NUM_EPOCHS),
    'RESUME_TARGET_EPOCHS'   : str(RESUME_TARGET_EPOCHS),
    'ES_PATIENCE'            : str(ES_PATIENCE),
    'ES_MIN_DELTA'           : str(ES_MIN_DELTA),
    'ES_WARMUP'              : str(ES_WARMUP),
    'INFERENCE_STEP_SIZE'    : str(INFERENCE_STEP_SIZE),
    'INFERENCE_DISABLE_TTA'  : INFERENCE_DISABLE_TTA,
    'INFERENCE_DEVICE'       : INFERENCE_DEVICE,
    'PP_THRESHOLDS'          : PP_THRESHOLDS,
    'NSD_TOLERANCE_MM'       : str(NSD_TOLERANCE_MM),
    'LOW_DICE_THRESHOLD'     : str(LOW_DICE_THRESHOLD),
    'BOOTSTRAP_N'            : str(BOOTSTRAP_N),
    'CUDA_VISIBLE_DEVICES'   : CUDA_VISIBLE_DEVICES,
    'nnUNet_n_proc_DA'       : NNUNET_N_PROC_DA,
    'nnUNet_compile'         : NNUNET_COMPILE,
    'EXPERIMENT_NAME'        : 'BraTSMENRT_baseline',
    'KAGGLE_OUTPUT_DIR'      : DRIVE_OUTPUT,
    'PYTHONUNBUFFERED'       : '1',
})

for d in ['nnunet_raw','nnunet_preprocessed','nnunet_results',
          'metrics','results','visualizations','inference_outputs','logs','checkpoints']:
    Path(f'{PROJECT}/{d}').mkdir(parents=True, exist_ok=True)
Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

print('Environment configured:')
print(f'  Dataset     : {DATASET_NAME} (ID={DATASET_ID})')
print(f'  Train cases : {MAX_TRAIN_CASES}')
print(f'  Folds       : {NUM_FOLDS}')
print(f'  Epochs/fold : {NNUNET_NUM_EPOCHS}')
print(f'  Trainer     : {NNUNET_TRAINER_CLASS}')
print(f'  Drive output: {DRIVE_OUTPUT}')

In [ ]:
# ── Cell 4: Clone Repository + Install Dependencies ───────────────────────────
import subprocess, sys
from pathlib import Path

if not Path('/content/nnunet/.git').exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/maryampervaiz-alt/nnunet-training.git',
                    '/content/nnunet'], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', '/content/nnunet', 'pull'], check=True)
    print('Repo updated.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                '/content/nnunet/requirements.txt', '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 5: Register Custom Trainer with nnU-Net ──────────────────────────────
import shutil, importlib.util
from pathlib import Path
import nnunetv2

nnunet_pkg_dir = Path(nnunetv2.__file__).parent
trainer_src    = Path('/content/nnunet/src/training/nnunet_trainer_es.py')
trainer_dst    = nnunet_pkg_dir / 'training' / 'nnUNetTrainer' / 'nnUNetTrainerEarlyStopping.py'

if not trainer_src.exists():
    raise FileNotFoundError(f'Trainer not found: {trainer_src}')

shutil.copy2(trainer_src, trainer_dst)
spec = importlib.util.spec_from_file_location('nnUNetTrainerEarlyStopping', trainer_dst)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
if not hasattr(mod, 'nnUNetTrainerEarlyStopping'):
    raise ImportError('Class not found in trainer file')

print('Custom trainer registered: OK')

In [ ]:
# ── Cell 6: Set Working Directory ─────────────────────────────────────────────
import os, sys

os.chdir('/content/nnunet')
if '/content/nnunet' not in sys.path:
    sys.path.insert(0, '/content/nnunet')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 7: STEP 1 — Prepare Dataset ─────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

if not Path(os.environ['TRAIN_SOURCE']).exists():
    raise FileNotFoundError(f'Data not found: {os.environ["TRAIN_SOURCE"]}')

print(f'Source : {os.environ["TRAIN_SOURCE"]}')
print(f'Cases  : {os.environ["MAX_TRAIN_CASES"]}')

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/01_prepare_dataset.py',
     '--train', os.environ['TRAIN_SOURCE'],
     '--skip-val', '--log-dir', 'logs',
     '--max-cases', os.environ['MAX_TRAIN_CASES']],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Data preparation failed (rc={proc.returncode})')
print('Dataset preparation complete.')

In [ ]:
# ── Cell 8: Verify Dataset Structure ─────────────────────────────────────────
import json, os
from pathlib import Path

dataset_dir = Path(os.environ['nnUNet_raw']) / 'Dataset001_BraTSMENRT'
images = sorted((dataset_dir / 'imagesTr').glob('*.nii.gz'))
labels = sorted((dataset_dir / 'labelsTr').glob('*.nii.gz'))
print(f'Images : {len(images)}')
print(f'Labels : {len(labels)}')
if len(images) == 0:
    raise RuntimeError('No images found — check Cell 7 output.')
print('Dataset structure OK.')

In [ ]:
# ── Cell 9: STEP 2 — nnU-Net Planning & Preprocessing ────────────────────────
import subprocess, sys, os

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/02_preprocess.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Preprocessing failed (rc={proc.returncode})')
print('Preprocessing complete.')

In [ ]:
# ── Cell 10: STEP 3 — Train All Folds ────────────────────────────────────────
# ~3-4 hours on T4. Checkpoints saved to Google Drive after every fold.

import os, subprocess, sys, shutil
from pathlib import Path

NUM_FOLDS    = int(os.environ['NUM_FOLDS'])
NUM_EPOCHS   = int(os.environ['NNUNET_NUM_EPOCHS'])
TRAINER      = os.environ['NNUNET_TRAINER_CLASS']
PLANS        = os.environ['NNUNET_PLANS_IDENTIFIER']
DRIVE_OUTPUT = os.environ['KAGGLE_OUTPUT_DIR']

def _ckpt_dir(fold):
    ds  = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
    return (Path(os.environ['nnUNet_results']) / ds
            / f"{TRAINER}__{PLANS}__{os.environ['NNUNET_CONFIGURATION']}" / f'fold_{fold}')

def _save_to_drive(fold):
    src = _ckpt_dir(fold)
    dst = Path(DRIVE_OUTPUT) / 'nnunet_results' / src.relative_to(os.environ['nnUNet_results'])
    dst.mkdir(parents=True, exist_ok=True)
    for fp in list(src.glob('*.pth')) + list(src.glob('training_log*.txt')):
        shutil.copy2(fp, dst / fp.name)
    mcsv = Path('metrics') / f'fold_{fold}_training.csv'
    if mcsv.exists():
        md = Path(DRIVE_OUTPUT) / 'metrics'
        md.mkdir(parents=True, exist_ok=True)
        shutil.copy2(mcsv, md / mcsv.name)
    print(f'  Fold {fold} saved to Drive.')

print(f'Training {NUM_FOLDS} folds x {NUM_EPOCHS} epochs')
fold_results = {}

for fold in range(NUM_FOLDS):
    print(f'\n{"="*64}\n  Fold {fold}/{NUM_FOLDS-1}\n{"="*64}')
    proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/03_train.py',
         '--folds', str(fold), '--num-epochs', str(NUM_EPOCHS),
         '--seed', os.environ['NNUNET_SEED'],
         '--trainer', TRAINER,
         '--es-patience', os.environ['ES_PATIENCE'],
         '--es-min-delta', os.environ['ES_MIN_DELTA'],
         '--es-warmup', os.environ['ES_WARMUP'],
         '--log-dir', 'logs', '--metrics-dir', 'metrics',
         '--checkpoints-dir', 'checkpoints'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    fold_results[fold] = proc.returncode
    print(f'Fold {fold}: {"OK" if proc.returncode == 0 else "FAILED"}')
    _save_to_drive(fold)

print(f'\n{"="*64}\nTraining summary:')
for fold, rc in fold_results.items():
    print(f'  Fold {fold}: {"OK" if rc == 0 else "FAILED"}')
print(f'Checkpoints saved to: {DRIVE_OUTPUT}')

In [ ]:
# ── Cell 11: Training History — All Folds ────────────────────────────────────
import pandas as pd, os
from pathlib import Path

NUM_FOLDS = int(os.environ['NUM_FOLDS'])
print('Per-fold training summary:')
print('=' * 70)
print(f"  {'Fold':>4}  {'Epochs':>6}  {'Best Dice':>10}  {'Final Dice':>11}  {'Final Loss':>11}")
print('  ' + '-' * 52)

for fold in range(NUM_FOLDS):
    csv = Path(f'metrics/fold_{fold}_training.csv')
    if not csv.exists():
        print(f'  {fold:>4}  (not trained)')
        continue
    hist = pd.read_csv(csv)
    n_ep       = len(hist)
    best_dice  = hist['val_dice'].max()     if 'val_dice'   in hist.columns else float('nan')
    final_dice = hist['val_dice'].iloc[-1]  if 'val_dice'   in hist.columns else float('nan')
    final_loss = hist['train_loss'].iloc[-1] if 'train_loss' in hist.columns else float('nan')
    print(f'  {fold:>4}  {n_ep:>6}  {best_dice:>10.4f}  {final_dice:>11.4f}  {final_loss:>11.4f}')
print('=' * 70)

In [ ]:
# ── Cell 12: Verify Checkpoints ───────────────────────────────────────────────
import os
from pathlib import Path

TRAINER   = os.environ['NNUNET_TRAINER_CLASS']
PLANS     = os.environ['NNUNET_PLANS_IDENTIFIER']
CONFIG    = os.environ['NNUNET_CONFIGURATION']
DS        = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
NUM_FOLDS = int(os.environ['NUM_FOLDS'])
results_dir = Path(os.environ['nnUNet_results'])

print('Checkpoint verification:')
print('=' * 60)
for fold in range(NUM_FOLDS):
    ckpt_dir = results_dir / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'
    if not ckpt_dir.exists():
        print(f'  Fold {fold}: NOT TRAINED'); continue
    final  = ckpt_dir / 'checkpoint_final.pth'
    best   = ckpt_dir / 'checkpoint_best.pth'
    f_sz = f'{final.stat().st_size/1e6:.1f} MB' if final.exists() else 'MISSING'
    b_sz = f'{best.stat().st_size/1e6:.1f} MB'  if best.exists()  else 'MISSING'
    print(f'  Fold {fold}: final={f_sz}  best={b_sz}')
print('=' * 60)

In [ ]:
# ── Cell 13: STEP 4 — Inference ───────────────────────────────────────────────
import os, subprocess, sys
from pathlib import Path

TRAINER   = os.environ['NNUNET_TRAINER_CLASS']
PLANS     = os.environ['NNUNET_PLANS_IDENTIFIER']
CONFIG    = os.environ['NNUNET_CONFIGURATION']
DS        = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
NUM_FOLDS = int(os.environ['NUM_FOLDS'])

def _has_checkpoint(fold):
    d = Path(os.environ['nnUNet_results']) / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'
    return (d / 'checkpoint_final.pth').exists() or (d / 'checkpoint_best.pth').exists()

trained_folds = [f for f in range(NUM_FOLDS) if _has_checkpoint(f)]
print(f'Folds with checkpoints: {trained_folds}')
if not trained_folds:
    raise RuntimeError('No trained folds found. Run Cell 10 first.')

for fold in trained_folds:
    print(f'\nInference fold {fold} ...')
    proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/04_inference.py',
         '--cv-mode', '--folds', str(fold),
         '--output', 'inference_outputs/cv', '--log-dir', 'logs'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    preds = list(Path(f'inference_outputs/cv/fold_{fold}').glob('*.nii.gz'))
    print(f'Fold {fold}: {len(preds)} predictions')

print('\nInference complete.')

In [ ]:
# ── Cell 14: Post-processing — CC Threshold Sweep ────────────────────────────
import numpy as np, nibabel as nib, os, pandas as pd
from scipy import ndimage
from pathlib import Path

THRESHOLDS = [int(x) for x in os.environ['PP_THRESHOLDS'].split(',')]
NUM_FOLDS  = int(os.environ['NUM_FOLDS'])
gt_dir = (Path(os.environ['nnUNet_raw'])
          / f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
          / 'labelsTr')

def cc_filter(arr, min_voxels):
    labeled, n = ndimage.label(arr > 0)
    if n == 0: return arr.copy().astype(np.uint8)
    sizes = ndimage.sum(arr > 0, labeled, range(1, n + 1))
    keep = np.zeros_like(labeled, dtype=bool)
    for idx, sz in enumerate(sizes, 1):
        if sz >= min_voxels: keep |= (labeled == idx)
    return keep.astype(np.uint8)

def mean_dice(pred_dir, gt_dir):
    preds = sorted(Path(pred_dir).glob('*.nii.gz'))
    if not preds: return float('nan')
    dices = []
    for pp in preds:
        gp = gt_dir / pp.name
        if not gp.exists(): continue
        p = np.asarray(nib.load(pp).dataobj, dtype=np.uint8)
        g = np.asarray(nib.load(gp).dataobj, dtype=np.uint8)
        tp = int(np.logical_and(p, g).sum())
        denom = int(p.sum()) + int(g.sum())
        dices.append(2 * tp / denom if denom > 0 else 1.0)
    return float(np.mean(dices)) if dices else float('nan')

trained_folds = [f for f in range(NUM_FOLDS) if Path(f'inference_outputs/cv/fold_{f}').exists()]
sweep_results = []
for thresh in THRESHOLDS:
    fold_dices = []
    for fold in trained_folds:
        src = Path(f'inference_outputs/cv/fold_{fold}')
        dst = Path(f'inference_outputs/cv/fold_{fold}_pp_{thresh}')
        dst.mkdir(parents=True, exist_ok=True)
        for pp in src.glob('*.nii.gz'):
            img = nib.load(pp)
            cleaned = cc_filter(np.asarray(img.dataobj, dtype=np.uint8), thresh)
            nib.save(nib.Nifti1Image(cleaned, img.affine, img.header), dst / pp.name)
        fold_dices.append(mean_dice(dst, gt_dir))
    sweep_results.append({'min_voxels': thresh, 'mean_DSC': float(np.nanmean(fold_dices))})
    print(f'  thresh={thresh:>4}  mean_DSC={float(np.nanmean(fold_dices)):.4f}')

sweep_df = pd.DataFrame(sweep_results)
BEST_THRESH = int(sweep_df.loc[sweep_df['mean_DSC'].idxmax(), 'min_voxels'])
print(f'\nBest threshold: {BEST_THRESH} voxels')

for fold in trained_folds:
    src = Path(f'inference_outputs/cv/fold_{fold}')
    dst = Path(f'inference_outputs/cv/fold_{fold}_postproc')
    dst.mkdir(parents=True, exist_ok=True)
    for pp in src.glob('*.nii.gz'):
        img = nib.load(pp)
        cleaned = cc_filter(np.asarray(img.dataobj, dtype=np.uint8), BEST_THRESH)
        nib.save(nib.Nifti1Image(cleaned, img.affine, img.header), dst / pp.name)
    print(f'Fold {fold}: post-processing done')
print('Post-processing complete.')

In [ ]:
# ── Cell 15: STEP 5 — Evaluate ───────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/05_evaluate.py',
     '--cv-mode',
     '--results-dir',   'results',
     '--nsd-tolerance', os.environ['NSD_TOLERANCE_MM'],
     '--low-dice',      os.environ['LOW_DICE_THRESHOLD'],
     '--bootstrap-n',   os.environ['BOOTSTRAP_N'],
     '--show-rankings', '--latex', '--stat-test',
     '--log-dir', 'logs'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\nEvaluation complete. Output files:')
for f in sorted(Path('results').glob('*.csv')):
    print(f'  {f.name}')
for f in sorted(Path('results').glob('*.tex')):
    print(f'  {f.name}')

In [ ]:
# ── Cell 16: STEP 6 — Visualizations ─────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

subprocess.run(
    [sys.executable, 'scripts/06_visualize.py',
     '--all', '--cv-mode',
     '--results-dir', 'results',
     '--metrics-dir', 'metrics',
     '--output-dir',  'visualizations',
     '--n-cases', '15', '--log-dir', 'logs'],
    env=os.environ.copy(),
)
print('Visualization files:')
for p in sorted(Path('visualizations').rglob('*.png')):
    print(f'  {p}  ({p.stat().st_size//1024} KB)')

In [ ]:
# ── Cell 17: Quantitative Results + Show Visualizations ──────────────────────
import pandas as pd, numpy as np, os
from pathlib import Path
from IPython.display import Image, display

NUM_FOLDS = int(os.environ['NUM_FOLDS'])
_DISPLAY = {
    'dice': 'DSC', 'hd95': 'HD95 (mm)', 'hd': 'HD (mm)',
    'nsd': 'NSD', 'precision': 'Precision', 'recall': 'Recall',
}

def show_table(csv_path, title):
    p = Path(csv_path)
    if not p.exists(): print(f'{title}: not found'); return
    df = pd.read_csv(p)
    rows = []
    for col, label in _DISPLAY.items():
        if col not in df.columns: continue
        v = pd.to_numeric(df[col], errors='coerce').replace([float('inf'), float('-inf')], float('nan')).dropna()
        if v.empty: continue
        rows.append({'Metric': label,
                     'Mean +/- Std': f'{v.mean():.4f} +/- {v.std():.4f}',
                     'Median': f'{v.median():.4f}'})
    print(f'\n{title} (n={len(df)} cases)')
    print('=' * 60)
    print(pd.DataFrame(rows).set_index('Metric').to_string())
    print('=' * 60)

for fold in range(NUM_FOLDS):
    show_table(f'results/fold_{fold}_per_case.csv', f'Fold {fold}')
show_table('results/cv_combined.csv', 'All Folds Combined')

# Show plots
for plot in ['metrics_violin.png', 'metrics_boxplot.png', 'fold_comparison.png']:
    p = Path('visualizations') / plot
    if p.exists():
        print(f'\n{plot}')
        display(Image(str(p), width=900))

# Show training curves
for fold in range(NUM_FOLDS):
    p = Path(f'visualizations/training_curve_fold_{fold}.png')
    if p.exists():
        print(f'\nTraining curve — fold {fold}')
        display(Image(str(p), width=800))

---
## Checkpoints saved to Google Drive

Find them at: **Google Drive → nnunet_checkpoints**

To resume on university GPU:
```bash
# Download checkpoints
cp -r /content/drive/MyDrive/nnunet_checkpoints/nnunet_results ./nnunet_results

# Resume training to 50 epochs
python scripts/03_train.py --continue-training --folds 0 1 2 3 4 --num-epochs 50
```